In [1]:
import pandas as pd 
import numpy as np
from datetime import datetime, date

In [2]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = "api.allenneuraldynamics.org"
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(
   host=API_GATEWAY_HOST,
   database=DATABASE,
   collection=COLLECTION,
)
print(docdb_api_client._base_url)

https://api.allenneuraldynamics.org/v1/metadata_index/data_assets


In [16]:
aggregate = [
  {
    "$match": {
      "data_description.project_name": "V1 Deep Dive", 
      "name": {"$regex": "filtered"},
      "processing.data_processes": {
        "$elemMatch": {
          "name": "Filter NWB values",
        }
      }
  },
  }
  ,
  {
    "$project": {
      "name": 1, 
      "subject_id": "$data_description.subject_id",
      "genotype": "$subject.subject_details.genotype", 
      "date_of_birth": "$subject.subject_details.date_of_birth", 
      "sex": "$subject.subject_details.sex", 
      "session_time": "$acquisition.acquisition_start_time",
      "project_name": "$data_description.project_name", 
      "modality": "$data_description.modalities.name",
      "column": { "$arrayElemAt": ["$data_description.tags", 0] },
      "volume": { "$arrayElemAt": ["$data_description.tags", 1] }
    }
  },
]
    
records = docdb_api_client.aggregate_docdb_records(
    pipeline = aggregate,
)

In [18]:
records = [r for r in records if 'Electron microscopy' not in r['modality']]
df = pd.DataFrame(records)

df['session_date'] = df.apply(lambda x: datetime.fromisoformat(x['session_time']).date(), axis=1)
df['session_time'] = df.apply(lambda x: datetime.fromisoformat(x['session_time']).time(), axis=1)
df['date_of_birth'] = df.apply(lambda x: datetime.strptime(x['date_of_birth'], '%Y-%m-%d').date(), axis=1)
df['age'] = df.apply(lambda x: (x['session_date'] - x['date_of_birth']).days, axis=1)

df['column'] = df.apply(lambda x: int(x['column'].split(' ')[-1]), axis=1)
df['volume'] = df.apply(lambda x: int(x['volume'].split(' ')[-1]), axis=1)

df['golden_mouse'] = False
df.loc[df.subject_id=='409828', 'golden_mouse'] = True

order = ['project_name','_id','name','subject_id','golden_mouse','genotype','date_of_birth','sex','modality',
         'session_date','age','session_time','column','volume']
df = df[order]

df

,project_name,_id,name,subject_id,golden_mouse,genotype,date_of_birth,sex,modality,session_date,age,session_time,column,volume
0,V1 Deep Dive,98ce8fa1-97f2-40c3-879b-0e99db3abe81,409828_2018-11-29_13-42-04_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-11-29,149,13:42:04.672510,5,2
1,V1 Deep Dive,b1e7cf7c-bb52-42c9-9dde-54e7594b3ae2,409828_2018-12-13_15-10-05_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-12-13,163,15:10:05.562960,1,3
2,V1 Deep Dive,2cb93697-c954-4835-9c17-6deed1cf1fb5,409828_2018-12-14_13-14-42_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-12-14,164,13:14:42.634500,1,4
3,V1 Deep Dive,5aac054c-c0ce-4c19-94e9-8135f047fa62,409828_2018-12-14_14-47-35_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-12-14,164,14:47:35.097180,1,5
4,V1 Deep Dive,7f7ef7f3-90c4-45d6-b3a4-fa7bf4579be6,416296_2018-11-05_14-13-21_filtered_2026-04-09...,416296,False,Camk2a-tTA/wt;tetO-GCaMP6s/wt,2018-08-06,Female,"[Planar optical physiology, Behavior videos]",2018-11-05,91,14:13:21.053750,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,V1 Deep Dive,63783ce0-8d69-438e-8e35-73fb8fd5d8c7,438833_2019-03-25_15-10-31_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-03-25,114,15:10:31.659070,3,3
98,V1 Deep Dive,a18d9824-37aa-46ce-8ea6-f807f6795622,438833_2019-03-26_13-10-05_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-03-26,115,13:10:05.286770,4,2
99,V1 Deep Dive,c17510e6-31b7-4001-8788-fe8e8ed30269,438833_2019-03-26_14-44-46_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-03-26,115,14:44:46.601660,4,3
100,V1 Deep Dive,de0aad16-7644-443a-83dc-029bc7342354,438833_2019-03-27_11-33-05_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-03-27,116,11:33:05.464370,5,2


In [25]:
len(df[df.subject_id == '409828'].sort_values(by='session_date', ascending=False)) 

28

In [34]:
df[df.subject_id == '409828'].sort_values(by=['column', 'volume', 'session_date'], ascending=False)

,project_name,_id,name,subject_id,golden_mouse,genotype,date_of_birth,sex,modality,session_date,age,session_time,column,volume
53,V1 Deep Dive,c24b6f01-05d0-4721-ae23-d48030857403,409828_2018-12-12_10-53-44_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-12-12,162,10:53:44.705010,5,5
51,V1 Deep Dive,699e4dcf-70fc-4db2-b438-54f55c606027,409828_2018-12-11_16-12-21_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-12-11,161,16:12:21.396890,5,4
50,V1 Deep Dive,3c01dc6a-e872-4a1b-a54d-c4781cb5f44e,409828_2018-12-04_15-27-53_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-12-04,154,15:27:53.082660,5,3
0,V1 Deep Dive,98ce8fa1-97f2-40c3-879b-0e99db3abe81,409828_2018-11-29_13-42-04_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-11-29,149,13:42:04.672510,5,2
65,V1 Deep Dive,6592ac98-b4ac-4102-9237-8d9063817f15,409828_2018-11-26_11-16-25_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-11-26,146,11:16:25.373720,5,1
69,V1 Deep Dive,4a87c914-a62c-4b2a-91dd-de97523d9c70,409828_2018-12-11_14-40-36_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-12-11,161,14:40:36.449080,4,5
67,V1 Deep Dive,7dae9405-e14a-4647-b3bd-c2906696387b,409828_2018-12-03_15-56-23_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-12-03,153,15:56:23.139770,4,4
14,V1 Deep Dive,2149df96-789c-40d2-942c-988fc4c48d3d,409828_2018-12-03_14-25-24_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-12-03,153,14:25:24.632480,4,3
64,V1 Deep Dive,5690e017-7c61-4570-8128-182ffc1d36c6,409828_2018-11-21_10-56-07_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-11-21,141,10:56:07.266840,4,2
82,V1 Deep Dive,2f3d1e9f-4e29-4701-8da3-5695db905631,409828_2018-11-21_09-22-23_filtered_2026-04-16...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,Male,"[Planar optical physiology, Behavior videos]",2018-11-21,141,09:22:23.259970,4,1


In [29]:
len(df[df.subject_id == '416296'].sort_values(by='session_date', ascending=False)) 

25

In [35]:
df[df.subject_id == '416296'].sort_values(by=['column', 'volume', 'session_date'], ascending=False)

,project_name,_id,name,subject_id,golden_mouse,genotype,date_of_birth,sex,modality,session_date,age,session_time,column,volume
86,V1 Deep Dive,b87ab715-b5ad-45fc-adfc-150e6f18a17e,416296_2018-12-05_13-02-39_filtered_2026-04-09...,416296,False,Camk2a-tTA/wt;tetO-GCaMP6s/wt,2018-08-06,Female,"[Planar optical physiology, Behavior videos]",2018-12-05,121,13:02:39.575900,5,5
46,V1 Deep Dive,a9a080ed-3b22-4ab9-a390-a45c4d726468,416296_2018-12-05_11-31-44_filtered_2026-04-09...,416296,False,Camk2a-tTA/wt;tetO-GCaMP6s/wt,2018-08-06,Female,"[Planar optical physiology, Behavior videos]",2018-12-05,121,11:31:44.290780,5,4
35,V1 Deep Dive,92817871-dc1e-4638-9231-319beba07e30,416296_2018-12-04_13-45-25_filtered_2026-04-09...,416296,False,Camk2a-tTA/wt;tetO-GCaMP6s/wt,2018-08-06,Female,"[Planar optical physiology, Behavior videos]",2018-12-04,120,13:45:25.427340,5,3
43,V1 Deep Dive,22212e61-9a14-462f-8670-0d615d362198,416296_2018-12-04_12-17-47_filtered_2026-04-09...,416296,False,Camk2a-tTA/wt;tetO-GCaMP6s/wt,2018-08-06,Female,"[Planar optical physiology, Behavior videos]",2018-12-04,120,12:17:47.000730,5,2
7,V1 Deep Dive,aaf89272-aee4-4b77-97c7-0b9e0850a045,416296_2018-11-14_10-20-29_filtered_2026-04-09...,416296,False,Camk2a-tTA/wt;tetO-GCaMP6s/wt,2018-08-06,Female,"[Planar optical physiology, Behavior videos]",2018-11-14,100,10:20:29.147870,5,1
36,V1 Deep Dive,5e07581c-6d0d-41d6-97d8-aceebfaa4d92,416296_2018-12-07_13-19-39_filtered_2026-04-09...,416296,False,Camk2a-tTA/wt;tetO-GCaMP6s/wt,2018-08-06,Female,"[Planar optical physiology, Behavior videos]",2018-12-07,123,13:19:39.881600,4,5
34,V1 Deep Dive,496645d0-ef98-4e55-a199-810bf683d157,416296_2018-12-03_12-42-43_filtered_2026-04-09...,416296,False,Camk2a-tTA/wt;tetO-GCaMP6s/wt,2018-08-06,Female,"[Planar optical physiology, Behavior videos]",2018-12-03,119,12:42:43.916490,4,4
17,V1 Deep Dive,96b081b5-a74a-4b8e-b179-d1daff8c0066,416296_2018-12-03_11-11-50_filtered_2026-04-09...,416296,False,Camk2a-tTA/wt;tetO-GCaMP6s/wt,2018-08-06,Female,"[Planar optical physiology, Behavior videos]",2018-12-03,119,11:11:50.005770,4,3
33,V1 Deep Dive,10638d89-8804-4a2f-a1a1-627530da7250,416296_2018-11-30_14-41-49_filtered_2026-04-09...,416296,False,Camk2a-tTA/wt;tetO-GCaMP6s/wt,2018-08-06,Female,"[Planar optical physiology, Behavior videos]",2018-11-30,116,14:41:49.316520,4,2
54,V1 Deep Dive,37ca66cc-8f20-41fc-be0c-01989d390b6b,416296_2018-11-16_12-27-27_filtered_2026-04-09...,416296,False,Camk2a-tTA/wt;tetO-GCaMP6s/wt,2018-08-06,Female,"[Planar optical physiology, Behavior videos]",2018-11-16,102,12:27:27.887290,4,1


In [37]:
len(df[df.subject_id == '427836'].sort_values(by='session_date', ascending=False)) 

24

In [36]:
df[df.subject_id == '427836'].sort_values(by=['column', 'volume', 'session_date'], ascending=False)

,project_name,_id,name,subject_id,golden_mouse,genotype,date_of_birth,sex,modality,session_date,age,session_time,column,volume
47,V1 Deep Dive,3ded8d2c-481b-4d9a-ac09-9dbc9b387120,427836_2019-04-30_15-10-52_filtered_2026-04-09...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"[Planar optical physiology, Behavior videos]",2019-04-30,204,15:10:52.762930,5,5
59,V1 Deep Dive,54fb413b-689b-423e-9d1a-74cb12552a7e,427836_2019-04-25_13-49-39_filtered_2026-04-09...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"[Planar optical physiology, Behavior videos]",2019-04-25,199,13:49:39.163550,5,4
44,V1 Deep Dive,64864d19-03b7-4529-acbd-8853b1ad3458,427836_2019-04-25_12-16-58_filtered_2026-04-09...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"[Planar optical physiology, Behavior videos]",2019-04-25,199,12:16:58.240890,5,3
88,V1 Deep Dive,c4cf0319-da99-41aa-962b-d01a504ef3db,427836_2019-04-18_13-14-46_filtered_2026-04-09...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"[Planar optical physiology, Behavior videos]",2019-04-18,192,13:14:46.884560,5,2
42,V1 Deep Dive,63e07fd1-fd55-478d-abe8-0221215a487d,427836_2019-04-18_11-48-17_filtered_2026-04-09...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"[Planar optical physiology, Behavior videos]",2019-04-18,192,11:48:17.399500,5,1
81,V1 Deep Dive,382ceffb-75b7-4a29-930a-5da423ffa09a,427836_2019-04-29_15-02-53_filtered_2026-04-16...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"[Planar optical physiology, Behavior videos]",2019-04-29,203,15:02:53.992910,4,5
90,V1 Deep Dive,b47e2543-329c-4bb1-b506-8f0739d447f6,427836_2019-04-29_13-36-17_filtered_2026-04-09...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"[Planar optical physiology, Behavior videos]",2019-04-29,203,13:36:17.106870,4,4
89,V1 Deep Dive,d54521e9-8942-4401-a0ed-775a67288e09,427836_2019-04-24_13-06-45_filtered_2026-04-09...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"[Planar optical physiology, Behavior videos]",2019-04-24,198,13:06:45.257460,4,3
21,V1 Deep Dive,d5a76f58-b4a4-46cd-a9f9-1adcf60fefd9,427836_2019-04-17_14-40-45_filtered_2026-04-09...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"[Planar optical physiology, Behavior videos]",2019-04-17,191,14:40:45.546830,4,2
77,V1 Deep Dive,fb846747-f00b-4336-bc6f-c46ae8843e02,427836_2019-04-17_13-08-38_filtered_2026-04-09...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"[Planar optical physiology, Behavior videos]",2019-04-17,191,13:08:38.689270,4,1


In [28]:
len(df[df.subject_id == '438833'].sort_values(by='session_date', ascending=False))

25

In [38]:
df[df.subject_id == '438833'].sort_values(by=['column', 'volume', 'session_date'], ascending=False)

,project_name,_id,name,subject_id,golden_mouse,genotype,date_of_birth,sex,modality,session_date,age,session_time,column,volume
31,V1 Deep Dive,89f2d5e7-d631-488e-93b4-ca6c1191d759,438833_2019-04-01_16-10-33_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-04-01,121,16:10:33.869560,5,5
30,V1 Deep Dive,bddf33d3-17c7-4095-9649-92e5b712a866,438833_2019-04-01_14-39-42_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-04-01,121,14:39:42.230830,5,4
27,V1 Deep Dive,40fd8a56-ce65-42fb-815f-f39a46e4ebb9,438833_2019-03-27_14-24-07_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-03-27,116,14:24:07.041320,5,3
100,V1 Deep Dive,de0aad16-7644-443a-83dc-029bc7342354,438833_2019-03-27_11-33-05_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-03-27,116,11:33:05.464370,5,2
62,V1 Deep Dive,89a2b07f-a080-44a1-8262-f25b39563c1d,438833_2019-03-21_14-08-14_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-03-21,110,14:08:14.261290,5,1
29,V1 Deep Dive,9fd5eaca-802c-448a-901f-e2dcbf1b1d50,438833_2019-03-29_14-55-11_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-03-29,118,14:55:11.866780,4,5
28,V1 Deep Dive,065d3b4e-c12a-4bd8-b25d-b020ac671ff0,438833_2019-03-29_13-14-43_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-03-29,118,13:14:43.095060,4,4
99,V1 Deep Dive,c17510e6-31b7-4001-8788-fe8e8ed30269,438833_2019-03-26_14-44-46_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-03-26,115,14:44:46.601660,4,3
98,V1 Deep Dive,a18d9824-37aa-46ce-8ea6-f807f6795622,438833_2019-03-26_13-10-05_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-03-26,115,13:10:05.286770,4,2
61,V1 Deep Dive,48314eaf-ce8d-4f91-9d36-a8c683c0edde,438833_2019-03-20_15-17-34_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,Male,"[Planar optical physiology, Behavior videos]",2019-03-20,109,15:17:34.583240,4,1


In [21]:
df.to_csv('/data/metadata/V1DD_metadata.csv', index= False)